# Pre-Lab 5: Forced vibration
Submit **one Word or PDF document** on Canvas before your own laboratory section. Include answers A–E and the three requested figures from the [Pre-Lab 5 handout](https://ehsan-roohi.github.io/MIE402-Lab/docs/MIE402_PreLab5_Fall2026.pdf). Run the supplied code; no additional programming is required. These are illustrative parameters, not your measured laboratory values.

## Physical model
For a direct force: $m\ddot{x}+c\dot{x}+kx=F_0\cos(\omega t)$. Mass $m$ is in kg, damping $c$ in N s/m, stiffness $k$ in N/m, displacement $x$ in m, time $t$ in s, force amplitude $F_0$ in N, and drive angular frequency $\omega$ in rad/s. Impulse $J$ has units N s.
For the actual two-spring driven-end apparatus: $m\ddot{x}+c\dot{x}+(k_1+k_2)x=k_2 y$. Here $y$ is endpoint displacement [m], $k_1$ is fixed-side stiffness and $k_2$ is driven-side stiffness [N/m]. Input peak amplitude is $Y$ [m]. The brake acts against the stationary track.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
m, c, k, F0, omega, J = 1., 1., 100., 1., 4*np.pi, 1.
wn=np.sqrt(k/m); alpha=c/(2*m); wd=np.sqrt(wn**2-alpha**2); zeta=c/(2*np.sqrt(m*k))
t=np.arange(0,20.001,.01)
xi=J/(m*wd)*np.exp(-alpha*t)*np.sin(wd*t)
fig,ax=plt.subplots();ax.plot(t,xi);ax.set(xlabel='Time (s)',ylabel='Displacement (m)',title='Unit-impulse response');ax.grid();plt.show()
H=1/complex(k-m*omega**2,c*omega)
ss=F0*abs(H)*np.cos(omega*t+np.angle(H))
sol=solve_ivp(lambda t,z:[z[1],(F0*np.cos(omega*t)-c*z[1]-k*z[0])/m],[t[0],t[-1]],[.1,0],t_eval=t,rtol=1e-9,atol=1e-11)
fig,ax=plt.subplots();ax.plot(t,sol.y[0],label='Full response');ax.plot(t,ss,'--',label='Steady state');ax.set(xlabel='Time (s)',ylabel='Displacement (m)');ax.legend();ax.grid();plt.show()
k1,k2,Y=45.,55.,.01
w=np.logspace(-1,2,400);hy=k2/(k1+k2-m*w*w+1j*c*w)
fig,ax=plt.subplots();ax.semilogx(w,20*np.log10(abs(hy)));ax.set(xlabel='Drive angular frequency (rad/s)',ylabel='Displacement ratio X/Y (dB)');ax.grid();plt.show()
tt=np.arange(0,40.001,.01)
s=solve_ivp(lambda t,z:[z[1],(k2*Y*np.cos(omega*t)-c*z[1]-(k1+k2)*z[0])/m],[tt[0],tt[-1]],[0,0],t_eval=tt,rtol=1e-9,atol=1e-11)
q=tt>30;A=np.column_stack([np.cos(omega*tt[q]),np.sin(omega*tt[q]),np.ones(sum(q))]);b=np.linalg.lstsq(A,s.y[0,q],rcond=None)[0]
numeric_gain=np.hypot(b[0],b[1])/Y;theory_gain=abs(k2/complex(k1+k2-m*omega**2,c*omega))
relative_error=abs(numeric_gain/theory_gain-1)
assert relative_error<.002
print(f'fn={wn/(2*np.pi):.4f} Hz, fd={wd/(2*np.pi):.4f} Hz, zeta={zeta:.4f}')
print(f'State-equation / transfer-response relative difference: {relative_error:.3g}')


## Submit brief explanations
**A.** Define and calculate natural and damped frequencies and damping ratio.
**B.** Explain what doubling impulse J changes. Include the impulse figure.
**C.** Identify transient and steady response. Include the time comparison. Which frequency remains?
**D.** Include X/Y versus angular frequency. Explain its low-frequency limit and agreement between the transfer response and numerical state equations.
**E.** List the measurements needed at the bench. Convert 0.024 m peak-to-peak and 72 rpm. Explain how to detect an unsettled record.
Export plots or use Save image; avoid screenshots of the notebook interface.